# Yolov8s-pose 剪枝模型+蒸馏
* teacher: yolov8s-pose.pt原始模型->生成伪标签
* student: yolov8s-pose-prune-sp0.5->剪枝模型

# yolov8s-pose.pt生成老师的伪标签

In [1]:
import argparse
import os
import json
from pathlib import Path
from ultralytics import YOLO
from tqdm import tqdm
import cv2

def generate_pseudo_labels(teacher_weights, coco_json, image_dir, output_json, conf_thr=0.6, imgsz=640, device="0"):
    # 加载 teacher 模型
    model = YOLO(teacher_weights, task='pose')
    model.to(device)

    # 加载原始 COCO 注释
    with open(coco_json, "r") as f:
        coco_anno = json.load(f) # dict

    # 原始 annotations
    original_annotations = coco_anno["annotations"]

    # category 信息不变
    categories = coco_anno["categories"]

    # COCO image_id 映射
    image_id_map = {img["file_name"]: img["id"] for img in coco_anno["images"]} # {'000000397133.jpg': 397133, ...}

    pseudo_annotations = []
    ann_id = max([anno["id"] for anno in original_annotations]) + 1

    # 遍历训练集图像
    img_files = list(Path(image_dir).rglob("*.jpg"))
    print(f"Found {len(img_files)} images in {image_dir}.")

    for img_path in tqdm(img_files, desc="Generating pseudo labels"):
        img = cv2.imread(str(img_path))
        if img is None:
            continue

        h, w = img.shape[:2]
        file_name = img_path.name
        if file_name not in image_id_map:
            continue

        image_id = image_id_map[file_name] # 320696

        # 推理
        results = model.predict(source=str(img_path), imgsz=imgsz, conf=conf_thr, device=device, verbose=False)
        r = results[0]

        if r.keypoints is None or len(r.keypoints) == 0:
            continue

        # 提取关键点预测
        boxes = r.boxes.xyxy.cpu().numpy() # [n, 4] # n个人
        scores = r.boxes.conf.cpu().numpy() # [n]
        classes = r.boxes.cls.cpu().numpy().astype(int) # 0-->person
        kpts = r.keypoints.xy.cpu().numpy() # [n, 17, 2]
        kpts_conf = r.keypoints.conf.cpu().numpy() # [n,17]

        for idx in range(len(boxes)):
            score = float(scores[idx])
            if score < conf_thr:
                continue

            box = boxes[idx] #[4]
            kps = kpts[idx] # [17,2]
            kps_c = kpts_conf[idx] #[17]

            # 转 COCO 格式 bbox
            x1, y1, x2, y2 = box
            bbox_w = x2 - x1
            bbox_h = y2 - y1

            # 转 COCO 格式关键点
            keypoints = []
            for i in range(kps.shape[0]):
                x, y = kps[i]
                c = int(kps_c[i] > 0.1)  # 可见性阈值
                keypoints.extend([float(x), float(y), c])

            ann = {
                "id": ann_id,
                "image_id": image_id,
                "category_id": 1,  # person
                "bbox": [float(x1), float(y1), float(bbox_w), float(bbox_h)],
                "score": score,
                "area": float(bbox_w * bbox_h),
                "iscrowd": 0,
                "keypoints": keypoints,
                "num_keypoints": int(sum(kps_c > 0.1))
            }

            pseudo_annotations.append(ann)
            ann_id += 1

    # 合并伪标注
    coco_anno["annotations"].extend(pseudo_annotations)

    # 保存新的 COCO 标注文件
    os.makedirs(Path(output_json).parent, exist_ok=True)
    with open(output_json, "w") as f:
        json.dump(coco_anno, f)

    print(f"Saved pseudo-labeled annotations to {output_json}")
    print(f"Added {len(pseudo_annotations)} pseudo labels.")

In [19]:
teacher = "weights/yolov8s-pose.pt"
coco_json = "datasets/coco-pose/annotations/person_keypoints_train2017.json" # train
image_dir = "datasets/coco-pose/images/train2017"
output_json = "datasets/coco-pose/annotations/person_keypoints_train2017_pseudo.json"
conf_thr = 0.7
imgsz = 1088 # 1088最好
device = 'device'

In [2]:
img_files = list(Path("datasets/coco-pose/images/train2017").rglob("*.jpg"))

In [3]:
len(img_files)

56599

In [7]:
img_files = list(Path("../ultralytics/datasets/coco-pose/images/val2017").rglob("*.jpg"))

In [8]:
len(img_files)

2347

In [5]:
len(img_files)

2347

In [6]:
!wc -l datasets/coco-pose/val2017.txt

2346 datasets/coco-pose/val2017.txt


In [7]:
coco_anno["images"][0]

{'license': 4,
 'file_name': '000000397133.jpg',
 'coco_url': 'http://images.cocodataset.org/val2017/000000397133.jpg',
 'height': 427,
 'width': 640,
 'date_captured': '2013-11-14 17:02:52',
 'flickr_url': 'http://farm7.staticflickr.com/6116/6255196340_da26cf2c9e_z.jpg',
 'id': 397133}

In [8]:
image_id_map = {img["file_name"]: img["id"] for img in coco_anno["images"]}

In [9]:
image_id_map

{'000000397133.jpg': 397133,
 '000000037777.jpg': 37777,
 '000000252219.jpg': 252219,
 '000000087038.jpg': 87038,
 '000000174482.jpg': 174482,
 '000000403385.jpg': 403385,
 '000000006818.jpg': 6818,
 '000000480985.jpg': 480985,
 '000000458054.jpg': 458054,
 '000000331352.jpg': 331352,
 '000000296649.jpg': 296649,
 '000000386912.jpg': 386912,
 '000000502136.jpg': 502136,
 '000000491497.jpg': 491497,
 '000000184791.jpg': 184791,
 '000000348881.jpg': 348881,
 '000000289393.jpg': 289393,
 '000000522713.jpg': 522713,
 '000000181666.jpg': 181666,
 '000000017627.jpg': 17627,
 '000000143931.jpg': 143931,
 '000000303818.jpg': 303818,
 '000000463730.jpg': 463730,
 '000000460347.jpg': 460347,
 '000000322864.jpg': 322864,
 '000000226111.jpg': 226111,
 '000000153299.jpg': 153299,
 '000000308394.jpg': 308394,
 '000000456496.jpg': 456496,
 '000000058636.jpg': 58636,
 '000000041888.jpg': 41888,
 '000000184321.jpg': 184321,
 '000000565778.jpg': 565778,
 '000000297343.jpg': 297343,
 '000000336587.jpg': 

In [13]:
original_annotations[2]["id"]

183830

In [14]:
img_files = list(Path("datasets/coco-pose/images/val2017").rglob("*.jpg"))

In [15]:
len(img_files)

2347

In [16]:
img_files[0]

PosixPath('datasets/coco-pose/images/val2017/000000320696.jpg')

In [17]:
image_id_map['000000320696.jpg']

320696

In [18]:
model = YOLO(teacher)

In [20]:
results = model.predict(source=str('datasets/coco-pose/images/val2017/000000320696.jpg'), imgsz=imgsz, conf=conf_thr, device=device, verbose=False)

In [27]:
results[0].orig_img.shape

(427, 640, 3)

In [41]:
results[0].keypoints.conf.shape

torch.Size([1, 17])

In [38]:
results[0].boxes.cls.shape

torch.Size([1])

In [10]:
import json
import os
from tqdm import tqdm

# 路径配置
pseudo_json_path = "datasets/coco-pose/annotations/person_keypoints_val2017_pseudo.json"
labels_dir = "datasets/coco-pose/labels_distill/val2017"
images_dir = "datasets/coco-pose/images/val2017"

# 如果 labels 目录不存在，创建
os.makedirs(labels_dir, exist_ok=True)

# 加载伪标签 JSON
with open(pseudo_json_path, "r") as f:
    data = json.load(f)

# 读取图像尺寸
image_info = {img["id"]: img for img in data["images"]}

# 遍历 annotations 生成 YOLO 格式 txt
for ann in tqdm(data["annotations"], desc="Converting to YOLO labels"):
    image_id = ann["image_id"]
    img = image_info[image_id]
    img_w, img_h = img["width"], img["height"]

    # 检查关键点
    keypoints = ann["keypoints"]
    if len(keypoints) != 51:
        continue

    # bbox 转换为 YOLO 格式
    x, y, w, h = ann["bbox"]
    x_center = (x + w / 2) / img_w
    y_center = (y + h / 2) / img_h
    w /= img_w
    h /= img_h

    # 关键点归一化
    kpts = []
    for i in range(0, len(keypoints), 3):
        kx = keypoints[i] / img_w
        ky = keypoints[i + 1] / img_h
        v = keypoints[i + 2]
        kpts.extend([kx, ky, v])

    # 保存到对应文件
    image_filename = img["file_name"]
    label_filename = os.path.splitext(image_filename)[0] + ".txt"
    label_path = os.path.join(labels_dir, label_filename)

    with open(label_path, "a") as lf:
        lf.write(f"0 {x_center} {y_center} {w} {h} " + " ".join(map(str, kpts)) + "\n")

print(f"✅ 已更新 YOLO 标签到: {labels_dir}")


Converting to YOLO labels: 100%|███████| 15870/15870 [00:00<00:00, 29988.40it/s]

✅ 已更新 YOLO 标签到: datasets/coco-pose/labels_distill/val2017


# 蒸馏

In [14]:
# train_student.py
from ultralytics import YOLO
import os

# -----------------------------
# 配置
# -----------------------------
student_weights = "weights/yolov8s-pose-prune-sp0.5.pt"  # Student 模型
# data_path = "datasets/coco-pose"                          # COCO 格式数据集目录
data_yaml = "datasets/my-coco-pose.yaml"
save_dir = "runs/student_train"                            # 保存训练结果目录

epochs = 100       # 训练轮数
batch_size = 16    # Batch size
img_size = 1088     # 输入图片尺寸
device = "0"       # 使用 GPU id

# -----------------------------
# 创建保存目录
# -----------------------------
os.makedirs(save_dir, exist_ok=True)

# -----------------------------
# 加载 Student 模型
# -----------------------------
model = YOLO(student_weights)

# -----------------------------
# 开始训练
# -----------------------------
model.train(
    data=data_yaml,        # 数据集路径
    epochs=epochs,
    batch=batch_size,
    imgsz=img_size,
    device=device,
    save_dir=save_dir
)

print(f"训练完成，结果保存在 {save_dir}")

New https://pypi.org/project/ultralytics/8.3.186 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.162 🚀 Python-3.10.18 torch-2.4.1+cu121 CUDA:0 (NVIDIA L40, 45386MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets/my-coco-pose.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1088, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=weights/yolov8s-pose-prune-sp0.5.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train52, nbs=64, nms=False,

train: Scanning /data/xl/Projects/EdgeLite/datasets/coco-pose/labels/train2017..


KeyboardInterrupt: 

In [5]:
from ultralytics.utils.plotting import plot_results

In [ ]:
plot_results()